In [2]:
import pandas as pd

import warnings

warnings.filterwarnings('ignore')

In [3]:
file_path = "../data/processed/ev_clean_dataset_2025.csv"

df = pd.read_csv(file_path)

df.head()

,region_country,category,parameter,mode,powertrain,year,unit,value,aggregate_group
0,World,Projection-STEPS,EV stock,2 and 3 wheelers,BEV,2030,Vehicles,170000000.0,_World
1,World,Projection-STEPS,EV stock,Cars,BEV,2030,Vehicles,150000000.0,_World
2,China,Projection-STEPS,EV stock,2 and 3 wheelers,BEV,2030,Vehicles,91000000.0,Other
3,China,Projection-STEPS,EV stock,Cars,BEV,2030,Vehicles,82000000.0,Other
4,World,Projection-STEPS,EV stock,Cars,PHEV,2030,Vehicles,82000000.0,_World


CREATING HISTORICAL DATA DATAFRAME.

WE WANT ACTUAL OBSERVATIONS NOT PROJECTIONS

In [5]:
df_hist = df[df["category"] == "Historical"].copy()

REMOVING DATA BY KEEPING ONLY COUNTRIES DATA

In [6]:
remove_regions = [

    "World",
    "Europe",
    "North America",
    "Central and South America",
    "Asia Pacific",
    "Africa",
    "Middle East",
    "EU27",
    "Other Europe",
    "Middle East and Caspian"
]

df_hist = df_hist[
    ~df_hist["region_country"]
      .isin(remove_regions)
]

In [13]:
Target = df_hist[
    (df_hist["parameter"] == "EV sales share") &
    (df_hist["mode"] == "Cars") &
    (df_hist["powertrain"] == "EV")
].copy()

# selecting required columns

Target = Target[["region_country", "year", "value"]]

Target = Target.rename(columns = {
    "region_country": "country",
    "value": "ev_sales_share"
})

print(Target.shape)

(664, 3)


FEATURE SELECTION FOR MACHINE LEARNING

In [ ]:
"""
EV STOCK

EV stock exists separately for:
- BEV (Battery Electric Vehicle)
- PHEV (Plug-In Hybrid Electric Vehicle)
- FCEV (Fuel Cell Electric Vehicle)

To represent total EV stock within a country,
all powertrain types are aggregating.

Total EV Stock =
BEV + PHEV + FCEV
"""

'\nEV STOCK\n\nEV stock exists separately for:\n- BEV (Battery Electric Vehicle)\n- PHEV (Plug-In Hybrid Electric Vehicle)\n- FCEV (Fuel Cell Electric Vehicle)\n\nTo represent total EV stock within a country,\nall powertrain types are aggregating.\n\nTotal EV Stock =\nBEV + PHEV + FCEV\n'

In [16]:
ev_stock = df_hist[
    (df_hist["parameter"] == "EV stock") &
    (df_hist["mode"] == "Cars")
].copy()

ev_stock = (ev_stock.groupby(
    ["region_country", "year"]
)["value"]
.sum()
.reset_index()
)

ev_stock.columns = [
    "country",
    "year",
    "ev_stock"
]

print(ev_stock.shape)
ev_stock.head()

(536, 3)


,country,year,ev_stock
0,Australia,2011,49.0
1,Australia,2012,300.0
2,Australia,2013,590.0
3,Australia,2014,1880.0
4,Australia,2015,3600.0


In [17]:
"""
EV STOCK SHARE

EV Stock Share represents the percentage of
the vehicle fleet that is electric.

This variable is already
reported as a combined EV percentage and
does not require aggregation.
"""

'\nEV STOCK SHARE\n\nEV Stock Share represents the percentage of\nthe vehicle fleet that is electric.\n\nThis variable is already\nreported as a combined EV percentage and\ndoes not require aggregation.\n'

In [18]:
ev_stock_share = df_hist[
    (df_hist["parameter"] == "EV stock share") &
    (df_hist["mode"] == "Cars")
].copy()

ev_stock_share = ev_stock_share[
    ["region_country", "year", "value"]
]

ev_stock_share.columns = [
    "country",
    "year",
    "ev_stock_share"
]

print(ev_stock_share.shape)
ev_stock_share.head()

(535, 3)


,country,year,ev_stock_share
8609,Norway,2024,32.0
8676,Norway,2023,30.0
8821,Norway,2022,26.0
8989,Norway,2021,22.0
9206,Iceland,2023,18.0


In [19]:
"""
EV CHARGING INFRASTRUCTURE

Charging infrastructure is reported separately as:

1. Publicly available fast chargers
2. Publicly available slow chargers

To obtain a complete measure of infrastructure
availability, both categories are combined.

Total Charging Points =
Fast Charging Points + Slow Charging Points
"""

'\nEV CHARGING INFRASTRUCTURE\n\nCharging infrastructure is reported separately as:\n\n1. Publicly available fast chargers\n2. Publicly available slow chargers\n\nTo obtain a complete measure of infrastructure\navailability, both categories are combined.\n\nTotal Charging Points =\nFast Charging Points + Slow Charging Points\n'

In [20]:
charging = df_hist[df_hist["parameter"] == "EV charging points"].copy()

charging_total = (charging.groupby(
    ["region_country", "year"]
)["value"]
.sum()
.reset_index())

charging_total.columns = [
    "country",
    "year",
    "charging_points"
]

print(charging_total.shape)
charging_total.head()

(391, 3)


,country,year,charging_points
0,Australia,2017,480.0
1,Australia,2018,731.0
2,Australia,2019,1960.0
3,Australia,2020,2480.0
4,Australia,2021,2770.0


In [21]:
"""
to keep ALL rows from your Target dataset
 even if some countries/years do NOT have EV stock, 
 EV stock share, or charging data.
 """

master_df = Target.copy()

master_df = master_df.merge(
    ev_stock,
    on=["country", "year"],
    how="left"
)

master_df = master_df.merge(
    ev_stock_share,
    on=["country", "year"],
    how="left"
)

master_df = master_df.merge(
    charging_total,
    on=["country", "year"],
    how="left"
)


print(master_df.shape)
master_df.head()

(664, 6)


,country,year,ev_sales_share,ev_stock,ev_stock_share,charging_points
0,Norway,2024,92.0,960230.0,32.0,31000.0
1,Norway,2023,90.0,900290.0,30.0,28000.0
2,Norway,2022,89.0,790260.0,26.0,25000.0
3,Norway,2021,86.0,630230.0,22.0,19700.0
4,Norway,2020,75.0,490200.0,17.0,17300.0


In [23]:
missing_summary = (
    master_df
    .isnull()
    .sum()
    .to_frame("missing_count")
)

missing_summary.head()

,missing_count
country,0
year,0
ev_sales_share,0
ev_stock,140
ev_stock_share,140


In [24]:
missing_summary["missing_percent"] = (
    missing_summary["missing_count"]
    / len(master_df)
    * 100
)

missing_summary.sort_values(
    "missing_percent",
    ascending=False
)


,missing_count,missing_percent
charging_points,277,41.716867
ev_stock,140,21.084337
ev_stock_share,140,21.084337
country,0,0.000000
year,0,0.000000
ev_sales_share,0,0.000000


In [25]:
print(master_df.shape)

master_df.isnull().sum()

(664, 6)


country              0
year                 0
ev_sales_share       0
ev_stock           140
ev_stock_share     140
charging_points    277
dtype: int64

In [26]:
master_df.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)